In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import seaborn as sns

In [ ]:
# Load the dataset
data_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv'
print("===================Data URL==================")
print(data_url)
print()
print("===================DataFrame==================")
df = pd.read_csv(data_url)
print(df)

In [ ]:
# By default, it returns the first 5 rows. 
df.head()

In [ ]:
# Print summary
df.info()

In [ ]:
# Drop rows with missing values and select some features
df = df.dropna()

features = ['horsepower','weight','displacement','acceleration','cylinders']
target = 'mpg'

In [ ]:
# Print summary
df.info()

# Notes: `dropna()`, Index Behavior, and Memory Usage

## Q1: Why did only `horsepower` show 392 (not 398) before `dropna()`?

"Non-Null Count" = number of values in that column that are **not** `NaN`.

- All columns had 398 non-null values **except** `horsepower`, which had 392.
- That means **6 rows** had a missing value specifically in `horsepower`, while every other column was fully populated for all 398 rows.
- After `dropna()`, those 6 rows are removed entirely, so every column now shows 392 — there are no missing values left anywhere.

---

## Q2: Why "392 entries, 0 to 397"?

`dropna()` **deletes rows** but does **not renumber the index**.

- Original index: `RangeIndex` from 0 to 397 (398 labels, no gaps).
- After dropping the 6 rows with missing `horsepower`, the remaining rows **keep their original index labels**.
- So the labels still span 0 → 397, but 6 numbers in that range are simply missing (the ones that got dropped) → 392 actual rows, spread across a range of 398 possible labels.

Confirm with:
```python
df.index
# Index([0, 1, 2, 3, ..., 395, 396, 397], dtype='int64', length=392)
```

To reset to a clean 0-to-391 index:
```python
df = df.reset_index(drop=True)
```

---

## Q3: Why did memory usage go **up** after dropping rows?

This is about the **index**, not the data itself.

| Before | After |
|---|---|
| `RangeIndex` | `Index` (Int64Index) |

- A `RangeIndex` is virtually free to store — it's just `start`, `stop`, `step` (like Python's `range()`), regardless of how many rows it represents.
- Once rows are dropped **non-contiguously** (gaps in the middle), pandas can no longer represent the index as a simple range.
- It converts to a regular `Int64Index`, which must **explicitly store all 392 integers** in memory.

**Net effect:** the memory saved by removing 6 rows of data is smaller than the memory cost of switching from a "free" RangeIndex to an explicitly stored integer array → total memory usage goes **up** (28.1 KB → 30.6 KB).

---

## Key takeaway

 `dropna()` (and most row-filtering operations) preserve original index labels and can silently convert a cheap `RangeIndex` into a more expensive stored index. Follow up with `.reset_index(drop=True)` if you want a clean, memory-efficient index again.


In [ ]:
# Calculate correlation matrix
correlation_matrix = df[features+[target]].corr()

# Plot heatmap of feature importance

plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap for Feature Importance')
plt.show

In [ ]:
# Select features from the correlation matrix and use mpg as the target variable
X = df[['horsepower', 'weight', 'displacement', 'cylinders']].values
y = df['mpg'].values

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Create a Linear Regression model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

In [ ]:
# Evaluate the mode
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")

In [ ]:
# Plot the true vs predicted mpg
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, alpha=0.75, color='blue', label='Predicted vs True')

plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Prediction Line')


# Calculate and plot prediction errors
for (true_value, predicted_value) in zip(y_test, y_pred):
    plt.plot([true_value, true_value], [true_value, predicted_value], color='gray', linestyle='-', linewidth=0.5)


plt.xlabel("True MPG")
plt.ylabel("Predicted MPG")
plt.title("True MPG vs Predicted MPG with Prediction Errors")
plt.legend()
plt.grid(True)
plt.show()
df.info()